# Monitor de Riesgo Bancario Chileno
## Notebook 1: Pipeline de Datos — CMF + Banco Central de Chile

> **Objetivo:** Construir un pipeline reproducible que descargue, limpie e integre datos de riesgo de crédito del sistema bancario chileno desde dos fuentes oficiales:
> - **CMF Chile:** Indicadores de morosidad 90+ días, provisiones y cartera vencida por institución
> - **Banco Central de Chile (API BDE):** TPM, IMACEC, colocaciones totales y tipo de cambio

**Período de análisis:** 2015 – 2025  
**Frecuencia:** Mensual  
**Autor:** Krishna Bustos

---
## 0. Configuración e Importaciones

In [1]:
import os
import time
import requests
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

load_dotenv()
BCC_USER = os.getenv('BCC_USER')
BCC_PASS = os.getenv('BCC_PASS')

ROOT     = Path('..')
RAW_DIR  = ROOT / 'data' / 'raw'
PROC_DIR = ROOT / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

print('Librerias cargadas correctamente')
print(f'Credenciales BCC: {"OK" if BCC_USER else "No encontradas - revisa el archivo .env"}')

Librerias cargadas correctamente
Credenciales BCC: OK


---
## 1. API del Banco Central de Chile (BDE)

El Banco Central provee una API REST publica con registro gratuito.  
Documentacion: https://si3.bcentral.cl/Siete/ES/Siete/API

| Serie | Codigo BDE | Descripcion |
|-------|-----------|-------------|
| TPM | F022.TPM.TIN.D001.NO.Z.M | Tasa de Politica Monetaria (mensual) |
| IMACEC | F032.IMC.IND.Z.Z.EP18.Z.Z.0.M | IMACEC, índice empalmado base 2018 |
| Colocaciones totales | F022.COL.PRO.Z.Z.CLP.D| Colocaciones efectivas CLP diarias |
| IPC | F074.IPC.VAR.Z.Z.C.M | IPC variación mensual |
| Dolar observado | F073.TCO.PRE.Z.D | Tipo de cambio observado |

In [2]:
BDE_URL = 'https://si3.bcentral.cl/SieteRestWS/SieteRestWS.ashx'

SERIES_BCC = {
    'tpm'          : 'F022.TPM.TIN.D001.NO.Z.M',
    'imacec'       : 'F032.IMC.IND.Z.Z.EP18.Z.Z.0.M',
    'colocaciones' : 'F022.COL.PRO.Z.Z.CLP.D',
    'ipc'          : 'F074.IPC.VAR.Z.Z.C.M',
    'usd_obs'      : 'F073.TCO.PRE.Z.D',
}

def descargar_serie_bcc(codigo, fecha_ini='2015-01-01', fecha_fin=None):
    if fecha_fin is None:
        fecha_fin = datetime.today().strftime('%Y-%m-%d')
    params = {
        'user'       : BCC_USER,
        'pass'       : BCC_PASS,
        'function'   : 'GetSeries',
        'timeseries' : codigo,
        'firstdate'  : fecha_ini,
        'lastdate'   : fecha_fin,
    }
    resp = requests.get(BDE_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if data.get('Codigo') != 0:
        raise ValueError(f"Error API BCC: {data.get('Descripcion')}")
    obs = data['Series']['Obs']
    df = pd.DataFrame(obs)
    df.columns = ['fecha_str', 'valor', 'estado']
    df = df[df['estado'] == 'OK'].copy()
    df['fecha'] = pd.to_datetime(df['fecha_str'], format='%d-%m-%Y')
    df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
    return df[['fecha', 'valor']].dropna().reset_index(drop=True)


def descargar_todas_las_series(series, fecha_ini='2015-01-01'):
    dfs = []
    for nombre, codigo in series.items():
        print(f'  Descargando: {nombre}...', end=' ')
        try:
            df = descargar_serie_bcc(codigo, fecha_ini=fecha_ini)
            df = df.set_index('fecha').resample('ME').last().reset_index()
            df = df.rename(columns={'valor': nombre})
            dfs.append(df.set_index('fecha'))
            print(f'OK ({len(df)} obs)')
        except Exception as e:
            print(f'Error: {e}')
        time.sleep(0.5)
    return pd.concat(dfs, axis=1).reset_index()


print('Descargando series del Banco Central...')
df_bcc = descargar_todas_las_series(SERIES_BCC)
df_bcc.to_parquet(RAW_DIR / 'bcc_series.parquet', index=False)
print(f'Dataset BCC guardado: {df_bcc.shape}')
df_bcc.tail()

Descargando series del Banco Central...
  Descargando: tpm... OK (138 obs)
  Descargando: imacec... OK (137 obs)
  Descargando: colocaciones... OK (138 obs)
  Descargando: ipc... OK (138 obs)
  Descargando: usd_obs... OK (139 obs)
Dataset BCC guardado: (139, 6)


,fecha,tpm,imacec,colocaciones,ipc,usd_obs
134,2026-03-31,4.5,120.023380,208225.424486,1.0,931.57
135,2026-04-30,4.5,114.236768,208867.158271,1.3,901.76
136,2026-05-31,4.5,112.659492,210713.610143,0.2,892.89
137,2026-06-30,4.5,NaN,212474.083109,0.0,922.34
138,2026-07-31,NaN,NaN,NaN,NaN,928.99


---
## 2. Datos CMF — Morosidad del Sistema Bancario

La CMF publica mensualmente el Indicador de Morosidad 90+ dias por banco.  
Los archivos Excel se descargan automaticamente con `src/descargar_cmf.py`.

> Ejecutar primero: `python src/descargar_cmf.py`  
> Los archivos quedaran en `data/raw/cmf_morosidad/`

In [3]:
def _generar_datos_muestra_cmf():
    """
    Genera datos de muestra calibrados con valores historicos reales
    del sistema bancario chileno (2015-2025).
    Se usa si aun no se han descargado los archivos CMF.
    """
    np.random.seed(42)
    bancos = {
        'Banco de Chile'      : {'base_mora': 1.2, 'vol': 0.15},
        'Banco Santander'     : {'base_mora': 1.5, 'vol': 0.18},
        'BancoEstado'         : {'base_mora': 2.8, 'vol': 0.25},
        'Banco BCI'           : {'base_mora': 1.3, 'vol': 0.16},
        'Scotiabank Chile'    : {'base_mora': 2.1, 'vol': 0.22},
        'Banco Itau'          : {'base_mora': 1.8, 'vol': 0.20},
        'Banco Security'      : {'base_mora': 1.0, 'vol': 0.12},
        'Banco Falabella'     : {'base_mora': 4.5, 'vol': 0.40},
        'Banco Ripley'        : {'base_mora': 5.2, 'vol': 0.45},
        'Banco Internacional' : {'base_mora': 1.6, 'vol': 0.18},
    }
    fechas = pd.date_range('2015-01-01', '2025-01-01', freq='ME')
    shock = np.ones(len(fechas))
    for i, f in enumerate(fechas):
        if pd.Timestamp('2019-10-01') <= f <= pd.Timestamp('2020-03-01'):
            shock[i] = 1.25
        elif pd.Timestamp('2020-03-01') <= f <= pd.Timestamp('2021-06-01'):
            shock[i] = 1.60
        elif pd.Timestamp('2022-01-01') <= f <= pd.Timestamp('2023-06-01'):
            shock[i] = 1.35
    registros = []
    for banco, params in bancos.items():
        base = params['base_mora']
        vol  = params['vol']
        mora_total = np.abs(base * shock + np.random.normal(0, vol, len(fechas)))
        for i, fecha in enumerate(fechas):
            registros.append({
                'fecha'         : fecha,
                'banco'         : banco,
                'mora_comercial': mora_total[i] * 0.75,
                'mora_consumo'  : mora_total[i] * 1.40,
                'mora_vivienda' : mora_total[i] * 0.55,
                'mora_total'    : mora_total[i],
            })
    return pd.DataFrame(registros)


def parsear_archivo_cmf(archivo, fecha):
    """Parser robusto SBIF (2015-2018) + CMF pre/post IFRS-9 (2019-2021 / 2022+)."""
    df = pd.read_excel(archivo, header=None)

    prefijos_banco = (
        'Banco ', 'Scotiabank', 'HSBC', 'JP Morgan', 'Jp Morgan',
        'China Construction', 'Bank of', 'Tanner', 'BBVA', 'Citibank',
        'Itaú', 'Itau',
    )

    # Detecta en qué columna están los bancos (0 antes de 2022, 1 desde 2022)
    fila_inicio, col_banco = None, None
    for col in (0, 1):
        for i in range(len(df)):
            v = str(df.iloc[i, col]).strip()
            if v.startswith(prefijos_banco):
                fila_inicio, col_banco = i, col
                break
        if fila_inicio is not None:
            break

    if fila_inicio is None:
        raise ValueError('No se encontró fila de inicio de bancos')

    # Las columnas de datos quedan desplazadas el mismo offset que la columna de bancos
    off = col_banco
    col_total, col_com, col_con, col_viv = 2 + off, 3 + off, 5 + off, 6 + off

    registros = []
    for i in range(fila_inicio, len(df)):
        nombre = str(df.iloc[i, col_banco]).strip()
        if not nombre or nombre.lower() == 'nan':
            break
        if 'Sistema Bancario' in nombre or nombre.startswith(('Notas', '(', '*')):
            break
        if not nombre.startswith(prefijos_banco):
            continue

        registros.append({
            'fecha'         : fecha,
            'banco'         : nombre,
            'mora_total'    : pd.to_numeric(df.iloc[i, col_total], errors='coerce'),
            'mora_comercial': pd.to_numeric(df.iloc[i, col_com],   errors='coerce'),
            'mora_consumo'  : pd.to_numeric(df.iloc[i, col_con],   errors='coerce'),
            'mora_vivienda' : pd.to_numeric(df.iloc[i, col_viv],   errors='coerce'),
        })
    return pd.DataFrame(registros)



def parsear_morosidad_cmf(directorio):
    archivos = sorted(Path(directorio).glob('*.xlsx'))
    if not archivos:
        print('No se encontraron archivos Excel. Usando datos de muestra...')
        return _generar_datos_muestra_cmf()

    dfs, errores = [], []
    for archivo in archivos:
        try:
            partes = archivo.stem.split('_')              # ['morosidad', '2016-01']
            fecha  = pd.to_datetime(f"{partes[1]}-01")    # '2016-01-01'
            df     = parsear_archivo_cmf(archivo, fecha)
            if not df.empty:
                dfs.append(df)
            else:
                errores.append((archivo.name, 'sin filas'))
        except Exception as e:
            errores.append((archivo.name, str(e)))

    if errores:
        print(f'  {len(errores)} archivos con problemas (mostrando primeros 3):')
        for nombre, err in errores[:3]:
            print(f'    {nombre}: {err}')

    if not dfs:
        print('No se pudo parsear nada. Usando datos de muestra.')
        return _generar_datos_muestra_cmf()

    return pd.concat(dfs, ignore_index=True)



print('Procesando datos CMF...')
CMF_DIR = RAW_DIR / 'cmf_morosidad'
CMF_DIR.mkdir(exist_ok=True)
df_cmf = parsear_morosidad_cmf(CMF_DIR)
df_cmf.to_parquet(RAW_DIR / 'cmf_morosidad.parquet', index=False)
print(f'Dataset CMF: {df_cmf.shape}')
print(f'Bancos: {df_cmf["banco"].nunique()}')
print(f'Periodo: {df_cmf["fecha"].min().strftime("%Y-%m")} a {df_cmf["fecha"].max().strftime("%Y-%m")}')
df_cmf.head()

Procesando datos CMF...
Dataset CMF: (2206, 6)
Bancos: 25
Periodo: 2016-01 a 2026-04


,fecha,banco,mora_total,mora_comercial,mora_consumo,mora_vivienda
0,2016-01-01,Banco Bice,0.299136,0.293768,0.317968,0.324868
1,2016-01-01,"Banco Bilbao Vizcaya Argentaria, Chile",1.396027,0.664100,1.929157,2.262100
2,2016-01-01,Banco BTG Pactual Chile,0.000000,0.000000,NaN,NaN
3,2016-01-01,Banco Consorcio,1.063497,1.051262,2.193025,0.294548
4,2016-01-01,Banco de Chile,1.221029,1.166377,1.701253,1.066174


---
## 3. Integracion de Fuentes — Dataset Maestro

In [4]:
BANCOS_RETAIL = {
    'Banco de Chile', 'Banco Santander', 'BancoEstado',
    'Banco BCI', 'Scotiabank Chile', 'Banco Itau',
    'Banco Falabella', 'Banco Ripley', 'Banco Security',
    'Banco Internacional', 'Banco Consorcio', 'Banco BICE',
    'Banco Bilbao Vizcaya Argentaria, Chile',
}

def construir_dataset_maestro(df_cmf, df_bcc):
    df_cmf = df_cmf.copy()
    df_cmf['fecha'] = pd.to_datetime(df_cmf['fecha']).dt.to_period('M').dt.to_timestamp('M')
    df_bcc = df_bcc.copy()
    df_bcc['fecha'] = pd.to_datetime(df_bcc['fecha']).dt.to_period('M').dt.to_timestamp('M')

    df_master = df_cmf.merge(df_bcc, on='fecha', how='left')
    df_master = df_master.sort_values(['banco', 'fecha'])

    # Clasificacion por tipo de banco
    df_master['tipo_banco'] = df_master['banco'].apply(
        lambda b: 'Retail' if b in BANCOS_RETAIL else 'Corporativo/Inversion'
    )

    # Flags de cartera disponible (minimo 6 meses con datos)
    df_master['tiene_cartera_consumo'] = df_master.groupby('banco')['mora_consumo'].transform(
        lambda x: x.notna().sum() >= 6
    )
    df_master['tiene_cartera_vivienda'] = df_master.groupby('banco')['mora_vivienda'].transform(
        lambda x: x.notna().sum() >= 6
    )

    # Variacion mensual de colocaciones — variable del sistema, misma para todos los bancos
    var_col_sistema = (
        df_master.drop_duplicates('fecha')
        .set_index('fecha')['colocaciones']
        .sort_index()
        .pct_change() * 100
    )
    df_master['var_colocaciones_m'] = df_master['fecha'].map(var_col_sistema)

    # Variacion anual de morosidad — SIN inf
    mora_yoy = df_master.groupby('banco')['mora_total'].pct_change(12) * 100
    df_master['mora_total_yoy'] = mora_yoy.replace([np.inf, -np.inf], np.nan)

    # Spread vs sistema
    mora_sistema = df_master.groupby('fecha')['mora_total'].transform('mean')
    df_master['mora_spread_vs_sistema'] = df_master['mora_total'] - mora_sistema

    # Ciclo macroeconomico
    condiciones = [
        df_master['fecha'].between('2015-01-01', '2019-09-30'),
        df_master['fecha'].between('2019-10-01', '2020-02-28'),
        df_master['fecha'].between('2020-03-01', '2021-05-31'),
        df_master['fecha'].between('2021-06-01', '2023-07-31'),
        df_master['fecha'] >= '2023-08-01',
    ]
    etiquetas = [
        'Pre-Estallido (2015-2019)',
        'Estallido Social',
        'COVID-19',
        'Ciclo Inflacionario / Alza TPM',
        'Normalizacion Monetaria',
    ]
    df_master['ciclo'] = np.select(condiciones, etiquetas, default='Otro')
    return df_master.reset_index(drop=True)


df_master = construir_dataset_maestro(df_cmf, df_bcc)
df_master.to_parquet(PROC_DIR / 'dataset_maestro.parquet', index=False)
print(f'Dataset maestro guardado: {df_master.shape}')
print(f'Columnas: {list(df_master.columns)}')
print()
print('Distribucion por tipo de banco:')
print(df_master.groupby('tipo_banco')['banco'].nunique().rename('n_bancos').to_string())
print()
print('Mora promedio por tipo de banco:')
print(df_master.groupby('tipo_banco')[['mora_total','mora_consumo','mora_vivienda']].mean().round(3).to_string())

Dataset maestro guardado: (2206, 18)
Columnas: ['fecha', 'banco', 'mora_total', 'mora_comercial', 'mora_consumo', 'mora_vivienda', 'tpm', 'imacec', 'colocaciones', 'ipc', 'usd_obs', 'tipo_banco', 'tiene_cartera_consumo', 'tiene_cartera_vivienda', 'var_colocaciones_m', 'mora_total_yoy', 'mora_spread_vs_sistema', 'ciclo']

Distribucion por tipo de banco:
tipo_banco
Corporativo/Inversion    17
Retail                    8

Mora promedio por tipo de banco:
                       mora_total  mora_consumo  mora_vivienda
tipo_banco                                                    
Corporativo/Inversion       1.259         1.444          1.922
Retail                      2.417         2.141          3.039


---
## 4. Validacion del Pipeline

In [5]:
print('=' * 55)
print('      REPORTE DE CALIDAD DEL PIPELINE')
print('=' * 55)
for col in df_master.columns:
    nulos = df_master[col].isnull().sum()
    pct   = nulos / len(df_master) * 100
    estado = 'OK' if pct < 5 else 'ADVERTENCIA' if pct < 20 else 'CRITICO'
    print(f'  {estado:<12} {col:<35} {nulos:>5} nulos ({pct:.1f}%)')
print('-' * 55)
print(f'  Total registros : {len(df_master):,}')
print(f'  Bancos unicos   : {df_master["banco"].nunique()}')
print(f'  Meses cubiertos : {df_master["fecha"].nunique()}')
print(f'  Periodo         : {df_master["fecha"].min().strftime("%b %Y")} a {df_master["fecha"].max().strftime("%b %Y")}')
print('=' * 55)

      REPORTE DE CALIDAD DEL PIPELINE
  OK           fecha                                   0 nulos (0.0%)
  OK           banco                                   0 nulos (0.0%)
  ADVERTENCIA  mora_total                            118 nulos (5.3%)
  ADVERTENCIA  mora_comercial                        118 nulos (5.3%)
  CRITICO      mora_consumo                          546 nulos (24.8%)
  CRITICO      mora_vivienda                         703 nulos (31.9%)
  OK           tpm                                     0 nulos (0.0%)
  OK           imacec                                  0 nulos (0.0%)
  OK           colocaciones                            0 nulos (0.0%)
  OK           ipc                                     0 nulos (0.0%)
  OK           usd_obs                                 0 nulos (0.0%)
  OK           tipo_banco                              0 nulos (0.0%)
  OK           tiene_cartera_consumo                   0 nulos (0.0%)
  OK           tiene_cartera_vivienda             